# Solution C1 · Perfect separation: when the logistic becomes very sure and wrong

**Type**: Break the model

## Objective

Build a case in which the classes are **perfectly separable on the training set** (few examples, many features), observe that without regularization the weights explode and the probabilities
become extreme, and show that the **AUC does not notice** while the log-loss does. Then find `C` with cross-validation.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate():
    X, y = make_classification(n_samples=5060, n_features=60, n_informative=5, n_redundant=0, class_sep=1.0, flip_y=0.05, random_state=1)
    return X[:60], y[:60], X[60:], y[60:]


X_tr, y_tr, X_te, y_te = generate()
Cs = np.logspace(-4, 6, 11)

## Commented solution

In [ ]:
rows = []
for c in Cs:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=20000)).fit(X_tr, y_tr)
    p_te = np.clip(m.predict_proba(X_te)[:, 1], 1e-15, 1 - 1e-15)
    confident = (p_te < 0.01) | (p_te > 0.99)
    wrong = (p_te > 0.5) != y_te
    rows.append({"train_acc": m.score(X_tr, y_tr), "l1_norm": np.abs(m[-1].coef_).sum(),
                 "train_logloss": log_loss(y_tr, m.predict_proba(X_tr)[:, 1], labels=[0, 1]),
                 "test_logloss": log_loss(y_te, p_te), "test_auc": roc_auc_score(y_te, p_te),
                 "confident_share": confident.mean(), "confident_errors": int((confident & wrong).sum())})
table = pd.DataFrame(rows, index=Cs)
table.round(3)

In [ ]:
best_test_C = float(table["test_logloss"].idxmin())
search = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=20000)), {"logisticregression__C": Cs},
                      scoring="neg_log_loss", cv=StratifiedKFold(5, shuffle=True, random_state=0)).fit(X_tr, y_tr)
cv_C = float(search.best_params_["logisticregression__C"])
auc_range = table["test_auc"].max() - table["test_auc"].min()
logloss_range = table["test_logloss"].max() - table["test_logloss"].min()
print(f"best C on the test set: {best_test_C} | C chosen in cross-validation: {cv_C} (test log-loss {table.loc[cv_C, 'test_logloss']:.3f} against {table.loc[Cs[-1], 'test_logloss']:.3f} without penalty)")
print(f"range of the test AUC: {auc_range:.3f} | range of the test log-loss: {logloss_range:.2f}")

## Check

In [ ]:
with verify():
    check(list(table.columns) == ["train_acc", "l1_norm", "train_logloss", "test_logloss", "test_auc", "confident_share", "confident_errors"], "table with the seven columns", "check names and order")
    check(table.loc[Cs[-1], "train_acc"] == 1.0 and table.loc[Cs[0], "train_acc"] < 0.7, "without regularization the training set is perfectly separated", "train_acc with C = 1e6 must be 1")
    check(table["l1_norm"].is_monotonic_increasing, "the norm of the weights always grows with C", "more C, less penalty, more weights")
    check(table.loc[Cs[-1], "train_logloss"] < 1e-3 and table.loc[Cs[-1], "test_logloss"] > 5, "training: log-loss almost 0; test: over 5", "train_logloss and test_logloss at the row C = 1e6")
    check(table.loc[Cs[-1], "confident_errors"] > 1000, "over 1000 examples out of 5000 are wrong with probability below 1% or above 99%", "count (p < 0.01 or p > 0.99) and (predicted class != y)")
    check(best_test_C == table["test_logloss"].idxmin() and best_test_C <= 0.1, "the best C on the test set is small (strong regularization)", "idxmin of the test log-loss")
    check(isinstance(cv_C, float) and cv_C <= 0.1 and table.loc[cv_C, "test_logloss"] < 0.75, "the C chosen in cross-validation clearly beats the unpenalized logistic on the test set", "GridSearchCV with scoring='neg_log_loss'")
    check(auc_range < 0.06 and logloss_range > 6, "the test AUC varies little (less than 0.06), the log-loss by more than 6", "compare the two ranges")
done()

## What you learned

- With more features than examples the training set is **always** separable: `train_acc = 1` is not a success, it is a property of the geometry. The training log-loss goes to zero and the
  weights grow without limit: the logistic without penalty has no solution. That is why scikit-learn's `LogisticRegression` is regularized by default.
- The "confident" predictions are an artifact: with `C = 1e6` over 1700 test examples out of 5000 are wrong with a probability below 1% or above 99%. **The test log-loss
  goes from 0.67 (the best model) to 7**: worse than a model that always says 50% (which is 0.693).
- **The AUC varies only from 0.62 to 0.66.** The ordering stays poorly informative in any case, and does not notice that the probabilities have become absurd. If the model feeds decisions
  with costs (exercise B2) or another estimate, look at the log-loss and the calibration too.
- `C` is chosen with cross-validation on a **probabilistic** metric (`neg_log_loss`): with the AUC as criterion the choice would have been almost indifferent and you would not have seen the
  problem. With 60 examples the choice is noisy: the value found is reasonable, not "the right one".

## Common pitfall

Raising `C` "to make the model converge better" or removing the regularization to "have pure coefficients". With small data or separable classes it produces enormous, meaningless
coefficients, and the solver stops because of the iteration limit, not because it found the minimum.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What happens to an unregularized logistic regression if the classes are linearly separable?</summary>

The likelihood has no finite maximum: as the norm of the weights increases the loss keeps decreasing toward zero, so the coefficients diverge and the solver stops only because of
tolerance or the maximum number of iterations. The probabilities become 0 or 1, the standard errors of the coefficients are nonsensical. It is solved with a penalty (L2, or L1 if I also want to
select) or with a Bayesian regularization, and the strength is chosen in cross-validation.

</details>